In [1]:
"""
Ablation Study 1: MSC-TimesNet Only (No STMAE / No Masking)
===========================================================
Modifications:
  - Bypasses STMAE pretraining and spatial topological Transformer encoding.
  - Channels & frequency bands are mapped via linear projection directly into MSC-TimesNet.
  - Retains identical data preprocessing (Per-Subject Z-Score, T=10, Stride=1),
    training dynamics, AdaBN recalibration, and 15-fold LOSO evaluation (Seed=42).
"""

import os
import time
import math
import random
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# =====================================================================
# CONFIGURATION & PATHS
# =====================================================================
DATA_SEARCH_PATHS = [
    "/kaggle/input/datasets/daviderusso7/seed-dataset",
    "/kaggle/input/seed-dataset",
    "/kaggle/input/daviderusso7/seed-dataset",
    "."
]
OUTPUT_DIR = "/kaggle/working/seed_ablation_no_stmae"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_CHANNELS = 62
NUM_CLASSES = 3         # Negative (0), Neutral (1), Positive (2)
RAW_DIM = 5             # 5 DE frequency bands: Delta, Theta, Alpha, Beta, Gamma
CLASS_NAMES = ["Negative", "Neutral", "Positive"]

# ---- Stage B : Sequences (T=10, Stride=1) ----
WINDOW_LENGTH = 10
STRIDE = 1

# ---- Stage C : MSC-TimesNet ----
EMBEDDING_SIZE = 32
CLASSIFIER_HIDDEN_SIZE = 128
CLASSIFIER_BLOCKS = 2
TOP_FREQUENCIES = 3
CLASSIFIER_FEEDFORWARD_SIZE = 256
CLASSIFIER_HEADS = 4
CLASSIFIER_DROPOUT = 0.3

# ---- Training Dynamics ----
MAX_EPOCHS = 50
MIN_EPOCHS = 15
PATIENCE_EPOCHS = 10
LEARNING_RATE = 1e-3
BATCH_SIZE = 128
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 5
GRADIENT_CLIP = 1.0
LABEL_SMOOTHING = 0.05
VALIDATION_FRACTION = 0.2
MIXUP_STRENGTH = 0.2
CHANNEL_DROPOUT_RATE = 0.1
RECALIBRATE_BATCHNORM = True

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

REPORT = []


def seed_everything(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =====================================================================
# DATA LOADING & NORMALIZATION
# =====================================================================
def find_npz_files():
    for p in DATA_SEARCH_PATHS:
        f_data = os.path.join(p, "DatasetCaricatoNoImage.npz")
        f_labels = os.path.join(p, "LabelsNoImage.npz")
        f_subs = os.path.join(p, "SubjectsNoImage.npz")
        if os.path.exists(f_data) and os.path.exists(f_labels) and os.path.exists(f_subs):
            return f_data, f_labels, f_subs
    raise FileNotFoundError("Could not find SEED NPZ files. Verify dataset path.")


def load_seed_npz():
    f_data, f_labels, f_subs = find_npz_files()
    print(f"  [+] Loading raw arrays from {os.path.dirname(f_data)}...")

    X = np.load(f_data)["arr_0"].astype(np.float32)
    Y = np.load(f_labels)["arr_0"].astype(np.int64)
    subs = np.load(f_subs)["arr_0"].astype(np.int32)

    # Transpose (N, 5, 62) to (N, 62, 5)
    if X.shape[1] == RAW_DIM and X.shape[2] == NUM_CHANNELS:
        X = np.transpose(X, (0, 2, 1))

    trial_ids = np.zeros(len(Y), dtype=np.int32)
    t_id = 0
    for i in range(1, len(Y)):
        if subs[i] != subs[i - 1] or Y[i] != Y[i - 1]:
            t_id += 1
        trial_ids[i] = t_id

    return X, Y, subs, trial_ids


def normalize_per_subject_zscore(X_raw, subject_ids):
    X_norm = np.empty_like(X_raw)
    for s in np.unique(subject_ids):
        idx = np.where(subject_ids == s)[0]
        sub_feats = X_raw[idx].reshape(-1, RAW_DIM).astype(np.float32)
        mu = sub_feats.mean(axis=0, keepdims=True)
        sd = sub_feats.std(axis=0, keepdims=True)
        sd[sd < 1e-8] = 1.0
        X_norm[idx] = ((sub_feats - mu) / sd).reshape(len(idx), NUM_CHANNELS, RAW_DIM)
    return X_norm


def build_sequence_index(y, subs, tri, seq_len=WINDOW_LENGTH, stride=STRIDE):
    keys = subs * 100000 + tri
    seqs, labs, s_sub, s_tri = [], [], [], []
    order = np.argsort(keys, kind="stable")
    for k in np.unique(keys):
        rows = order[keys[order] == k]
        if rows.shape[0] < seq_len:
            continue
        for start in range(0, rows.shape[0] - seq_len + 1, stride):
            win = rows[start:start + seq_len]
            seqs.append(win)
            labs.append(y[win[0]])
            s_sub.append(subs[win[0]])
            s_tri.append(tri[win[0]])
    return (np.asarray(seqs, dtype=np.int64), np.asarray(labs, dtype=np.int64),
            np.asarray(s_sub, dtype=np.int64), np.asarray(s_tri, dtype=np.int64))


# =====================================================================
# MSC-TIMESNET MODEL (WITHOUT STMAE)
# =====================================================================
def fft_topk_periods(x, k=TOP_FREQUENCIES):
    B, T, d = x.shape
    xf = torch.fft.rfft(x, dim=1)
    amp = xf.abs().mean(dim=2)
    amp[:, 0] = 0.0
    k = min(k, max(amp.shape[1] - 1, 1))
    _, idx = torch.topk(amp, k, dim=1)
    freqs = idx.float().mean(dim=0).round().long().clamp(min=1)
    periods = [max(int(T // f.item()), 1) for f in freqs]
    weights = torch.stack([amp[:, i] for i in freqs], dim=1)
    return periods, F.softmax(weights, dim=1)


class MultiScaleConvBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        h = max(d_model // 4, 8)
        self.b1 = nn.Sequential(nn.Conv2d(d_model, h, 1), nn.BatchNorm2d(h), nn.GELU())
        self.b3 = nn.Sequential(nn.Conv2d(d_model, h, 3, padding=1), nn.BatchNorm2d(h), nn.GELU())
        self.b5 = nn.Sequential(nn.Conv2d(d_model, h, 5, padding=2), nn.BatchNorm2d(h), nn.GELU())
        self.bp = nn.Sequential(
            nn.AvgPool2d(3, stride=1, padding=1), nn.Conv2d(d_model, h, 1),
            nn.BatchNorm2d(h), nn.GELU(),
        )
        self.fuse = nn.Sequential(nn.Conv2d(4 * h, d_model, 1), nn.BatchNorm2d(d_model))

    def forward(self, x):
        return self.fuse(torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1))


class TimesBlock(nn.Module):
    def __init__(self, d_model, topk=TOP_FREQUENCIES):
        super().__init__()
        self.topk = topk
        self.conv = MultiScaleConvBlock(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        B, T, d = x.shape
        periods, weights = fft_topk_periods(x, self.topk)
        outs = []
        for p in periods:
            pad = (math.ceil(T / p) * p) - T
            xp = F.pad(x, (0, 0, 0, pad)) if pad > 0 else x
            Tp = xp.shape[1]
            num_p = Tp // p
            z = xp.permute(0, 2, 1).reshape(B, d, num_p, p)
            z = self.conv(z)
            z = z.reshape(B, d, Tp).permute(0, 2, 1)[:, :T, :]
            outs.append(z)
        stacked = torch.stack(outs, dim=-1)
        w = weights.unsqueeze(1).unsqueeze(1)
        agg = (stacked * w).sum(dim=-1)
        return self.norm(agg + x)


class MSCTimesNet(nn.Module):
    def __init__(self, in_dim, d_model=CLASSIFIER_HIDDEN_SIZE, blocks=CLASSIFIER_BLOCKS,
                 num_classes=NUM_CLASSES, dropout=CLASSIFIER_DROPOUT):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(in_dim, d_model), nn.LayerNorm(d_model))
        self.blocks = nn.ModuleList([TimesBlock(d_model) for _ in range(blocks)])
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=CLASSIFIER_HEADS, dim_feedforward=CLASSIFIER_FEEDFORWARD_SIZE,
            dropout=dropout, batch_first=True, norm_first=True, activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=1)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Dropout(dropout), nn.Linear(d_model, num_classes)
        )

    def forward(self, x):
        h = self.inp(x)
        for blk in self.blocks:
            h = blk(h)
        h = self.transformer(h)
        return self.head(h.mean(dim=1))


class AblationTimesNetOnly(nn.Module):
    """Replaces STMAE spatial encoder with a direct linear projection per electrode."""
    def __init__(self, in_channels=NUM_CHANNELS, raw_dim=RAW_DIM, emb_dim=EMBEDDING_SIZE,
                 num_classes=NUM_CLASSES):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(raw_dim, emb_dim),
            nn.LayerNorm(emb_dim)
        )
        self.net = MSCTimesNet(in_channels * emb_dim, num_classes=num_classes)

    def forward(self, x):
        # x shape: (B, T, C, RAW_DIM)
        B, T, C, Fq = x.shape
        h = self.proj(x)                           # (B, T, C, emb_dim)
        h = h.reshape(B, T, C * EMBEDDING_SIZE)    # Flatten channels for TimesNet
        return self.net(h)


# =====================================================================
# FAST GPU UTILITIES
# =====================================================================
def balanced_weights(y):
    cnt = np.bincount(y, minlength=NUM_CLASSES).astype(np.float32)
    cnt[cnt == 0] = 1.0
    w = cnt.sum() / (NUM_CLASSES * cnt)
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def lr_at(ep, base_lr, warmup_epochs=WARMUP_EPOCHS):
    warmup_epochs = max(1, int(warmup_epochs))
    if ep <= warmup_epochs:
        return base_lr * ep / warmup_epochs
    prog = (ep - warmup_epochs) / max(1, MAX_EPOCHS - warmup_epochs)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * min(prog, 1.0)))


def grouped_split(labels, groups, frac=VALIDATION_FRACTION, seed=RANDOM_SEED):
    gss = GroupShuffleSplit(n_splits=1, test_size=frac, random_state=seed)
    tr, va = next(gss.split(np.zeros(len(labels)), labels, groups))
    return tr, va


def make_batches(n, batch, shuffle=True):
    idx = np.random.permutation(n) if shuffle else np.arange(n)
    for i in range(0, n, batch):
        yield idx[i:i + batch]


def apply_channel_dropout(xb, p=CHANNEL_DROPOUT_RATE):
    if p <= 0:
        return xb
    B = xb.shape[0]
    keep = (torch.rand(B, 1, NUM_CHANNELS, 1, device=xb.device) > p).float()
    return xb * keep


def mixup(xb, yb, alpha=MIXUP_STRENGTH):
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(xb.shape[0], device=xb.device)
    return lam * xb + (1 - lam) * xb[perm], yb, yb[perm], lam


@torch.no_grad()
def adabn_recalibrate(model, X_pt, seq_idx_pt, rows_pt, batch=BATCH_SIZE):
    had_bn = any(isinstance(m, nn.BatchNorm2d) for m in model.modules())
    if not had_bn:
        return model
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.reset_running_stats()
            m.momentum = None
            m.train()
    for b_rows in make_batches(len(rows_pt), batch, shuffle=False):
        xb = X_pt[seq_idx_pt[rows_pt[b_rows]]]
        model(xb)
    model.eval()
    return model


@torch.no_grad()
def predict_probs(model, X_pt, seq_idx_pt, rows_pt, batch=BATCH_SIZE):
    model.eval()
    out = []
    for b_rows in make_batches(len(rows_pt), batch, shuffle=False):
        xb = X_pt[seq_idx_pt[rows_pt[b_rows]]]
        out.append(F.softmax(model(xb), dim=1).cpu().numpy())
    return np.concatenate(out, axis=0)


def fit_ablation(X_pt, seq_idx_pt, labels, groups, train_rows, seed):
    seed_everything(seed)
    model = AblationTimesNetOnly().to(DEVICE)
    tr_loc, va_loc = grouped_split(labels[train_rows], groups[train_rows], seed=seed)

    tr_rows_pt = torch.tensor(train_rows[tr_loc], dtype=torch.long, device=DEVICE)
    va_rows_pt = torch.tensor(train_rows[va_loc], dtype=torch.long, device=DEVICE)

    w = balanced_weights(labels[train_rows[tr_loc]])
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=LABEL_SMOOTHING)
    opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    best_f1, best_state, bad = -1.0, None, 0
    for ep in range(1, MAX_EPOCHS + 1):
        cur = lr_at(ep, LEARNING_RATE)
        for g in opt.param_groups:
            g["lr"] = cur

        model.train()
        for b_rows in make_batches(len(tr_rows_pt), BATCH_SIZE):
            b_idx = tr_rows_pt[b_rows]
            xb = X_pt[seq_idx_pt[b_idx]]
            yb = torch.tensor(labels[tr_rows_pt[b_rows].cpu().numpy()], dtype=torch.long, device=DEVICE)

            xb = apply_channel_dropout(xb)
            xb, ya, ybb, lam = mixup(xb, yb)
            logits = model(xb)
            loss = lam * crit(logits, ya) + (1 - lam) * crit(logits, ybb)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            opt.step()

        vp = predict_probs(model, X_pt, seq_idx_pt, va_rows_pt)
        vf1 = f1_score(labels[va_rows_pt.cpu().numpy()], vp.argmax(1), average="macro")
        if vf1 > best_f1:
            best_f1, bad = vf1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if ep >= MIN_EPOCHS and bad >= PATIENCE_EPOCHS:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_f1


# =====================================================================
# EVALUATION & SOFT TRIAL VOTING
# =====================================================================
def trial_level_scores(probs, y, sub, tri):
    keys = sub * 100000 + tri
    yt, yp = [], []
    for k in np.unique(keys):
        m = (keys == k)
        true_label = np.bincount(y[m]).argmax()
        pred_label = probs[m].mean(axis=0).argmax()
        yt.append(true_label)
        yp.append(pred_label)

    yt, yp = np.array(yt), np.array(yp)
    t_acc = accuracy_score(yt, yp)
    t_bacc = balanced_accuracy_score(yt, yp)
    t_macro_f1 = f1_score(yt, yp, average="macro", zero_division=0)
    t_weighted_f1 = f1_score(yt, yp, average="weighted", zero_division=0)
    return t_acc, t_bacc, t_macro_f1, t_weighted_f1, len(yt)


def run_fold(fold_name, X_pt, seq_idx_pt, packs, train_rows, test_rows):
    labels, s_sub, s_tri = packs[1:]
    groups = s_sub * 100000 + s_tri
    te_rows_pt = torch.tensor(test_rows, dtype=torch.long, device=DEVICE)

    model, _ = fit_ablation(X_pt, seq_idx_pt, labels, groups, train_rows, seed=RANDOM_SEED)

    if RECALIBRATE_BATCHNORM:
        model = adabn_recalibrate(model, X_pt, seq_idx_pt, te_rows_pt)
    probs = predict_probs(model, X_pt, seq_idx_pt, te_rows_pt)

    # Window-level Metrics
    yt = labels[test_rows]
    pred_win = probs.argmax(1)
    win_acc = accuracy_score(yt, pred_win)
    win_bacc = balanced_accuracy_score(yt, pred_win)
    win_macro_f1 = f1_score(yt, pred_win, average="macro", zero_division=0)
    win_weighted_f1 = f1_score(yt, pred_win, average="weighted", zero_division=0)

    # Trial-level Metrics
    t_acc, t_bacc, t_macro_f1, t_weighted_f1, num_trials = trial_level_scores(
        probs, yt, s_sub[test_rows], s_tri[test_rows]
    )

    print(
        f"  -> {fold_name:<11} | "
        f"WIN: acc={win_acc:.4f} bacc={win_bacc:.4f} macF1={win_macro_f1:.4f} wF1={win_weighted_f1:.4f} | "
        f"TRIAL: acc={t_acc:.4f} bacc={t_bacc:.4f} macF1={t_macro_f1:.4f} wF1={t_weighted_f1:.4f} "
        f"({len(test_rows)}w / {num_trials}t)"
    )

    REPORT.append(dict(
        fold=fold_name,
        win_acc=win_acc, win_bacc=win_bacc, win_macro_f1=win_macro_f1, win_weighted_f1=win_weighted_f1,
        trial_acc=t_acc, trial_bacc=t_bacc, trial_macro_f1=t_macro_f1, trial_weighted_f1=t_weighted_f1,
        num_windows=len(test_rows), num_trials=num_trials
    ))


def main():
    t0 = time.time()
    seed_everything(RANDOM_SEED)
    print("device:", DEVICE)
    print("Ablation Mode: NO STMAE (Linear Projection -> MSC-TimesNet)")

    print("\n[1] Loading SEED dataset from NPZ files...")
    X_raw, y, subs, tri = load_seed_npz()
    print(f"shape={X_raw.shape}  labels={np.bincount(y)}")
    print("subjects:", sorted(np.unique(subs).tolist()))

    print("\n[2] Normalizing (Two-Tier: Per-Subject Z-Score across 5 frequency bands)...")
    X = normalize_per_subject_zscore(X_raw, subs)

    print("\n[2.5] Pushing dataset to GPU VRAM...")
    X_pt = torch.tensor(X, dtype=torch.float32, device=DEVICE)

    print("\n[3] Skipping STMAE Pretraining...")

    print("\n[4] Building sliding sequence windows (T=10, Stride=1)...")
    packs = build_sequence_index(y, subs, tri, WINDOW_LENGTH, stride=STRIDE)
    seq_idx = packs[0]
    seq_idx_pt = torch.tensor(seq_idx, dtype=torch.long, device=DEVICE)
    print(f"  {seq_idx.shape[0]} windows  labels={np.bincount(packs[1])}")

    # -----------------------------------------------------------------
    # LEAVE-ONE-SUBJECT-OUT (LOSO) EVALUATION
    # -----------------------------------------------------------------
    print("\n" + "=" * 84)
    print("EVALUATION: ABLATION 1 (NO STMAE) -- SEED 3-CLASS LOSO (SEED = 42)")
    print("=" * 84)

    s_sub = packs[2]
    unique_subs = np.unique(s_sub)
    print(f"  loso_subject       {len(unique_subs)} folds x 1 seed = {len(unique_subs)} fits")

    for sb in unique_subs:
        te_rows = np.where(s_sub == sb)[0]
        tr_rows = np.where(s_sub != sb)[0]
        fold_name = f"subject_{sb}"
        run_fold(fold_name, X_pt, seq_idx_pt, packs, tr_rows, te_rows)

    df = pd.DataFrame(REPORT)
    results_path = os.path.join(OUTPUT_DIR, "results_ablation1_no_stmae_loso.csv")
    df.to_csv(results_path, index=False)
    print(f"\nsaved: {results_path}")

    print("\n" + "=" * 84)
    print("SUMMARY -- ABLATION 1 (NO STMAE): SEED 3-CLASS LOSO (Single-Seed = 42)")
    print("=" * 84)
    if not df.empty:
        summary_dict = {
            "win_acc": [df["win_acc"].mean()],
            "win_bacc": [df["win_bacc"].mean()],
            "win_macro_f1": [df["win_macro_f1"].mean()],
            "win_weighted_f1": [df["win_weighted_f1"].mean()],
            "trial_acc": [df["trial_acc"].mean()],
            "trial_bacc": [df["trial_bacc"].mean()],
            "trial_macro_f1": [df["trial_macro_f1"].mean()],
            "trial_weighted_f1": [df["trial_weighted_f1"].mean()],
            "folds": [len(df)]
        }
        summ = pd.DataFrame(summary_dict)
        print(summ.to_string(index=False, float_format=lambda v: "%.4f" % v))

    print(f"\nWall time: {(time.time() - t0) / 60.0:.1f} min")


if __name__ == "__main__":
    main()

device: cuda
Ablation Mode: NO STMAE (Linear Projection -> MSC-TimesNet)

[1] Loading SEED dataset from NPZ files...
  [+] Loading raw arrays from /kaggle/input/datasets/daviderusso7/seed-dataset...
shape=(50910, 62, 5)  labels=[16800 16560 17550]
subjects: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

[2] Normalizing (Two-Tier: Per-Subject Z-Score across 5 frequency bands)...

[2.5] Pushing dataset to GPU VRAM...

[3] Skipping STMAE Pretraining...

[4] Building sliding sequence windows (T=10, Stride=1)...
  49155 windows  labels=[16260 15885 17010]

EVALUATION: ABLATION 1 (NO STMAE) -- SEED 3-CLASS LOSO (SEED = 42)
  loso_subject       15 folds x 1 seed = 15 fits
  -> subject_0   | WIN: acc=0.6839 bacc=0.6840 macF1=0.6893 wF1=0.6911 | TRIAL: acc=0.6154 bacc=0.6167 macF1=0.6339 wF1=0.6313 (3277w / 13t)
  -> subject_1   | WIN: acc=0.7370 bacc=0.7408 macF1=0.7333 wF1=0.7311 | TRIAL: acc=0.7692 bacc=0.7500 macF1=0.7435 wF1=0.7562 (3277w / 13t)
  -> subject_2   | WIN: acc=0.5490 bacc